In [2]:

import numpy as np
import matplotlib as mpl
mpl.use('Qt5Agg')  # Use Qt5 backend for GUI to work properly
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import sys
sys.path.insert(0, r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\code")


import pickle
import os
import pandas as pd
import kaleido
import plotly.graph_objects as go
import plotly.io as pio

# Force the engine setting
#pio.kaleido.scope.default_format = "svg"
from plotly.subplots import make_subplots
from AnalasysFunction import Split_cal,VolToCalIdx,LongLIST,splitTrace_from_arrays,correct_spikes,_edit_spikes_gui,splitTrace,motorSp,plotVolCal,CalInt,CS_detection,remove_Frame_Multi
from spike_detection_Qixixn2 import complex_bursts_detection,refine_single_spikes,spike_height_calculation,detect_complex_spikes,refine_all_spikes,plot_trace_with_spikes_pdf,plot_trace_with_spikes_export,plot_trace_with_spikes_html
# Create dictionary
import pickle
from matplotlib.widgets import Slider, Button
from qixin_spike_detection_4 import sst_spike_correction_gui

In [3]:
#load data
path = r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW\28-10-2025-motor\fov6\Sync\cal\suite2p'
p_path = os.path.dirname(path)

# TracePathCal = os.path.join(path,'calTraceDF.csv')
# TracePathVol = os.path.join(path,'volTraceDF.csv')
# TracePathSPIKE = os.path.join(path,'SpikeIdx.csv')
# VolTrace = pd.read_csv(TracePathVol)
# VolTrace = np.array(VolTrace)
# VolTrace = VolTrace.flatten()
# Trace = VolTrace
# CalTrace = pd.read_csv(TracePathCal)
# CalTrace = np.array(CalTrace)
# CalTrace = CalTrace.flatten()
# VolAX = np.linspace(0, (len(Trace)/500), len(Trace)) 
# TraceC = CalTrace
# CalAX = np.linspace(0, (len(TraceC)/30), len(TraceC))
# parentP = os.path.dirname(path)
# MotPath = os.path.join(parentP,'Sync','MotorId.csv')
# motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
# IntCalT,IntCalXax = CalInt(TraceC,CalAX)
# spikeId = pd.read_csv(TracePathSPIKE)
# spikeId = np.array(spikeId)
# spikeId = spikeId.flatten()
# spikeId = spikeId.tolist()
# motor_active = motor.any()
# motor = motor[0:np.size(Trace,0)]



In [5]:
# RAW->MEAN TIFF UTILS
import os
import re
import numpy as np
import tifffile


def _infer_hw_from_mean_tif(mean_tif_path):
    arr = tifffile.imread(mean_tif_path)
    if arr.ndim < 2:
        raise ValueError(f"Mean TIFF must be at least 2D: {mean_tif_path}")
    h, w = arr.shape[-2], arr.shape[-1]
    return int(h), int(w)


def _infer_hw_from_vol_header(vol_header_path):
    txt = open(vol_header_path, 'r', encoding='utf-8', errors='ignore').read()

    def _grab_int(field_name):
        # Example in your files: "ImageWidth": { ... "IV":"74" ... }
        pat = rf'"{re.escape(field_name)}"[\s\S]*?"IV"\s*:\s*"?([0-9]+)"?'
        m = re.search(pat, txt)
        return int(m.group(1)) if m else None

    w = _grab_int('ImageWidth')
    h = _grab_int('ImageHeight')
    framebytes = _grab_int('ImageFramebytes')
    return h, w, framebytes


def _recover_escaped_windows_path(p):
    """
    Recover a Windows path when the original Python string was not raw
    and escape sequences like 
, 
, 	 were interpreted as control chars.
    """
    p = str(p)
    ctrl_named = {
        '\r': r'\r',
        '\n': r'\n',
        '\t': r'\t',
        '\f': r'\f',
        '\b': r'\b',
        '\v': r'\v',
        '\a': r'\a',
    }
    out = []
    changed = False
    for ch in p:
        if ch in ctrl_named:
            out.append(ctrl_named[ch])
            changed = True
        elif ord(ch) < 32:
            out.append('\\' + format(ord(ch), '02o'))
            changed = True
        else:
            out.append(ch)
    return ''.join(out), changed


def save_mean_tiff_from_raw(
    raw_path,
    out_tif=None,
    *,
    height=None,
    width=None,
    vol_header_path=None,
    mean_tif_ref=None,
    dtype=np.uint16,
    little_endian=True,
    chunk_frames=2000,
    out_dtype=np.uint16,
):
    """
    Load a binary RAW movie, compute mean image across frames, and save as TIFF.

    Dimension inference order:
      1) explicit height/width
      2) vol_header_path -> ImageWidth/ImageHeight
      3) mean_tif_ref -> image shape
    """
    raw_path = os.path.abspath(str(raw_path))
    if not os.path.exists(raw_path):
        recovered_raw, changed = _recover_escaped_windows_path(raw_path)
        if changed and os.path.exists(recovered_raw):
            print(f"[WARN] Recovered escaped path: {raw_path} -> {recovered_raw}")
            raw_path = recovered_raw
        else:
            raise FileNotFoundError(
                f"RAW not found: {raw_path}\n"
                "Tip: use a raw string for Windows paths, e.g. path = r'Z:\...'."
            )

    h = int(height) if height is not None else None
    w = int(width) if width is not None else None

    framebytes_header = None
    if vol_header_path is not None:
        vol_header_path = os.path.abspath(str(vol_header_path))
        if not os.path.exists(vol_header_path):
            rec_h, ch_h = _recover_escaped_windows_path(vol_header_path)
            if ch_h and os.path.exists(rec_h):
                vol_header_path = rec_h

    if mean_tif_ref is not None:
        mean_tif_ref = os.path.abspath(str(mean_tif_ref))
        if not os.path.exists(mean_tif_ref):
            rec_m, ch_m = _recover_escaped_windows_path(mean_tif_ref)
            if ch_m and os.path.exists(rec_m):
                mean_tif_ref = rec_m

    if (h is None or w is None) and vol_header_path is not None and os.path.exists(vol_header_path):
        hh, ww, fb = _infer_hw_from_vol_header(vol_header_path)
        h = h if h is not None else hh
        w = w if w is not None else ww
        framebytes_header = fb

    if (h is None or w is None) and mean_tif_ref is not None and os.path.exists(mean_tif_ref):
        hh, ww = _infer_hw_from_mean_tif(mean_tif_ref)
        h = h if h is not None else hh
        w = w if w is not None else ww

    # Auto fallback: infer shape from TIFFs in same folder as RAW
    if h is None or w is None:
        raw_dir = os.path.dirname(raw_path)
        tif_candidates = [
            os.path.join(raw_dir, 'Mean.tif'),
            os.path.join(raw_dir, 'ChanA_Preview.tif'),
            os.path.join(raw_dir, 'ChanB_Preview.tif'),
        ]
        # append any other tif/tiff files
        try:
            for fn in os.listdir(raw_dir):
                low = fn.lower()
                if low.endswith('.tif') or low.endswith('.tiff'):
                    tif_candidates.append(os.path.join(raw_dir, fn))
        except Exception:
            pass

        seen = set()
        for cand in tif_candidates:
            cand = os.path.abspath(cand)
            if cand in seen:
                continue
            seen.add(cand)
            if not os.path.exists(cand):
                continue
            try:
                hh, ww = _infer_hw_from_mean_tif(cand)
                h = h if h is not None else hh
                w = w if w is not None else ww
                if h is not None and w is not None:
                    print(f"[INFO] Inferred RAW shape from TIFF reference: {cand} -> ({h}, {w})")
                    break
            except Exception:
                continue

    if h is None or w is None:
        raise ValueError('Could not infer RAW frame size. Provide height/width or vol_header_path or mean_tif_ref.')

    dt = np.dtype(dtype)
    dt = dt.newbyteorder('<' if little_endian else '>')

    mm = np.memmap(raw_path, dtype=dt, mode='r')
    pix_per_frame = int(h * w)
    if pix_per_frame <= 0:
        raise ValueError('Invalid frame size.')

    n_frames = int(mm.size // pix_per_frame)
    rem = int(mm.size % pix_per_frame)
    if n_frames == 0:
        raise ValueError('RAW file too small for one frame with given shape.')

    if rem != 0:
        print(f"[WARN] RAW has {rem} extra pixels beyond full frames; they will be ignored.")

    bytes_per_pixel = dt.itemsize
    calc_framebytes = pix_per_frame * bytes_per_pixel
    if framebytes_header is not None and framebytes_header != calc_framebytes:
        print(f"[WARN] Header ImageFramebytes={framebytes_header}, calculated={calc_framebytes}. Using calculated from h*w*dtype.")

    stack = mm[: n_frames * pix_per_frame].reshape((n_frames, h, w))

    # Chunked mean to keep memory stable
    acc = np.zeros((h, w), dtype=np.float64)
    for i in range(0, n_frames, int(chunk_frames)):
        block = np.asarray(stack[i:i + int(chunk_frames)], dtype=np.float64)
        acc += block.sum(axis=0)
    mean_img = acc / float(n_frames)

    if out_tif is None:
        out_tif = os.path.join(os.path.dirname(raw_path), 'Mean_from_raw.tif')

    if out_dtype is None:
        save_img = mean_img.astype(np.float32)
    else:
        odt = np.dtype(out_dtype)
        if np.issubdtype(odt, np.integer):
            info = np.iinfo(odt)
            save_img = np.clip(np.round(mean_img), info.min, info.max).astype(odt)
        else:
            save_img = mean_img.astype(odt)

    tifffile.imwrite(out_tif, save_img, photometric='minisblack')
    print(f"Saved mean TIFF: {out_tif}")
    print(f"Frames used: {n_frames}, shape: ({h}, {w}), dtype: {dt}")

    return mean_img, {
        'raw_path': raw_path,
        'out_tif': out_tif,
        'n_frames': n_frames,
        'height': h,
        'width': w,
        'dtype': str(dt),
        'remainder_pixels': rem,
    }



# Example usage (uncomment to run):
path = r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc46\RL\04-01-2025-anst\fov8\jedi-150um'
raw_path = os.path.join(path, 'Image_001_001.raw')
header_path = os.path.join(path, 'vol_header.txt')
mean_img, meta = save_mean_tiff_from_raw(raw_path, vol_header_path=header_path)
print(meta)


<>:92: DeprecationWarning: invalid escape sequence '\.'
<>:92: DeprecationWarning: invalid escape sequence '\.'
C:\Users\owner\AppData\Local\Temp\ipykernel_26424\2510958488.py:92: DeprecationWarning: invalid escape sequence '\.'
  "Tip: use a raw string for Windows paths, e.g. path = r'Z:\...'."


[INFO] Inferred RAW shape from TIFF reference: Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc46\RL\04-01-2025-anst\fov8\jedi-150um\ChanA_Preview.tif -> (48, 80)
Saved mean TIFF: Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc46\RL\04-01-2025-anst\fov8\jedi-150um\Mean_from_raw.tif
Frames used: 17513, shape: (48, 80), dtype: uint16
{'raw_path': 'Z:\\Adam-Lab-Shared\\Data\\Michal_Rubin\\rugc46\\RL\\04-01-2025-anst\\fov8\\jedi-150um\\Image_001_001.raw', 'out_tif': 'Z:\\Adam-Lab-Shared\\Data\\Michal_Rubin\\rugc46\\RL\\04-01-2025-anst\\fov8\\jedi-150um\\Mean_from_raw.tif', 'n_frames': 17513, 'height': 48, 'width': 80, 'dtype': 'uint16', 'remainder_pixels': 0}


In [10]:
import numpy as np
from pathlib import Path
path= r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\srugc27\L\13-01-2026-Anst\fov2\Sync\cal\suite2p\plane0'
suite2p_plane = os.path.join(path,r'\suite2p\plane0')
suite2p_plane = Path(r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\srugc27\L\13-01-2026-Anst\fov2\Sync\cal\suite2p\plane0")  
#suite2p_plane = Path(r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\srugc17\Xb\17-06-2025\fov1\")
ops_path = os.path.join(suite2p_plane,'ops.npy')
data_path = os.path.join(suite2p_plane,'data.bin')
# Load ops to get metadata
ops = np.load(ops_path, allow_pickle=True).item()

Ly = ops["Ly"]
Lx = ops["Lx"]
nframes = ops["nframes"]

# Memory-map the binary file (recommended for large files)
data = np.memmap(
    data_path,
    dtype="int16",
    mode="r",
    shape=(nframes, Ly, Lx)
)

print("Shape:", data.shape)

mean_image = np.mean(data, axis=0)

plt.imshow(mean_image, cmap='gray')
plt.savefig(os.path.join(path,'mean_image.png'))




Shape: (1921, 512, 512)


In [11]:
import tifffile

output_path = os.path.join(path,r'motion_corrected_full.tif')

tifffile.imwrite(
    output_path,
    data,
    photometric="minisblack"
)

print("Saved to:", output_path)


Saved to: Z:\Adam-Lab-Shared\Data\Michal_Rubin\srugc27\L\13-01-2026-Anst\fov2\Sync\cal\suite2p\plane0\motion_corrected_full.tif


In [32]:
import numpy as np
import plotly.graph_objects as go

# ----------------------------
# Helpers
# ----------------------------
def robust_z(x):
    """Robust z-score using median/MAD. Returns float array."""
    x = np.asarray(x, float).ravel()
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-12
    return (x - med) / (1.4826 * mad)

def build_zproxy_from_ops(ops, *, use_vcorr_if_possible=True, upsample_vcorr_if_needed=True):
    """
    Build per-frame feature matrix (Zmat) and a single z-proxy score (z_proxy)
    to flag likely z/defocus frames. Uses Suite2p ops fields.

    Returns
    -------
    t_frames : (n,) int
    Zmat     : (n, nfeat) float
    feat_names : list[str]
    z_proxy  : (n,) float
    badframes : (n,) bool
    """
    # --- master length: xoff/yoff are per-frame ---
    xoff = np.asarray(ops.get("xoff", ops.get("xoff1", [])), float).ravel()
    yoff = np.asarray(ops.get("yoff", ops.get("yoff1", [])), float).ravel()
    if xoff.size == 0 or yoff.size == 0:
        raise KeyError("Could not find per-frame xoff/yoff (or xoff1/yoff1) in ops.")

    n = int(min(xoff.size, yoff.size))
    xoff = xoff[:n]
    yoff = yoff[:n]
    motion_xy = np.sqrt(xoff**2 + yoff**2)

    # corrXY should be per-frame; fall back to corrXY1 if needed
    corrXY = ops.get("corrXY", ops.get("corrXY1", None))
    if corrXY is None:
        raise KeyError("Could not find corrXY or corrXY1 in ops.")
    corrXY = np.asarray(corrXY, float).ravel()[:n]

    # Suite2p badframes (if present)
    badframes = np.asarray(ops.get("badframes", np.zeros(n, dtype=bool))).astype(bool).ravel()
    if badframes.size != n:
        badframes = badframes[:n] if badframes.size > n else np.pad(badframes, (0, n - badframes.size), constant_values=False)

    # --- features (per frame) ---
    # z-like proxy: z motion/defocus tends to LOWER correlation-to-template, so invert correlations.
    feat_motion = robust_z(motion_xy)
    feat_corrXY_bad = robust_z(-corrXY)

    cols = [feat_motion, feat_corrXY_bad]
    feat_names = ["xy_motion_z", "corrXY_bad_z"]

    # --- optional Vcorr (may NOT be per frame in your case) ---
    if use_vcorr_if_possible and ("Vcorr" in ops):
        Vcorr = np.asarray(ops["Vcorr"], float).ravel()
        if Vcorr.size == n:
            cols.append(robust_z(-Vcorr))
            feat_names.append("Vcorr_bad_z")
        elif upsample_vcorr_if_needed and Vcorr.size > 2:
            # Upsample to per-frame using linear interpolation across the recording
            x_old = np.linspace(0, n - 1, Vcorr.size)
            x_new = np.arange(n)
            Vcorr_up = np.interp(x_new, x_old, Vcorr)
            cols.append(robust_z(-Vcorr_up))
            feat_names.append("Vcorr_bad_z_upsampled")
        else:
            print(f"Skipping Vcorr: length {Vcorr.size} != nframes {n} and upsampling disabled/insufficient.")

    Zmat = np.column_stack(cols)

    # --- combine into a single z-proxy score ---
    # Emphasize mismatch/defocus metrics more than xy motion.
    weights = np.ones(Zmat.shape[1], float)

    # downweight xy motion
    if "xy_motion_z" in feat_names:
        weights[feat_names.index("xy_motion_z")] = 0.2

    # upweight corr-related
    for nm in ["corrXY_bad_z", "Vcorr_bad_z", "Vcorr_bad_z_upsampled"]:
        if nm in feat_names:
            weights[feat_names.index(nm)] = 0.4

    weights = weights / weights.sum()
    z_proxy = Zmat @ weights

    t_frames = np.arange(n)
    return t_frames, Zmat, feat_names, z_proxy, badframes

def plot_zproxy(t, Zmat, feat_names, z_proxy, badframes=None, *, title_prefix=""):
    # Line plot
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=z_proxy, mode="lines", name="z_proxy (combined)"))

    for j, nm in enumerate(feat_names):
        fig.add_trace(go.Scatter(x=t, y=Zmat[:, j], mode="lines", name=nm, opacity=0.5))

    if badframes is not None and np.any(badframes):
        bf = np.asarray(badframes, bool)
        fig.add_trace(go.Scatter(
            x=t[bf], y=z_proxy[bf],
            mode="markers",
            name="ops['badframes']",
            marker=dict(size=5, symbol="x")
        ))

    fig.update_layout(
        title=(title_prefix + "Per-frame z-shift proxy (mismatch/defocus)"),
        xaxis_title="Frame",
        yaxis_title="Robust z-score / proxy",
        template="plotly_white",
    )

    # Heatmap (feature matrix)
    fig_hm = go.Figure(data=go.Heatmap(
        z=Zmat.T,
        x=t,
        y=feat_names,
    ))
    fig_hm.update_layout(
        title=(title_prefix + "Per-frame feature matrix (robust z-scored)"),
        xaxis_title="Frame",
        yaxis_title="Feature",
        template="plotly_white",
    )
    return fig, fig_hm

# ----------------------------
# Run (expects ops already loaded)
# ----------------------------
ops_path = os.path.join(p_path,r'Sync\cal\suite2p\plane0\ops.npy')
ops = np.load(ops_path, allow_pickle=True).item()

t, Zmat, feat_names, z_proxy, badframes = build_zproxy_from_ops(
    ops,
    use_vcorr_if_possible=True,
    upsample_vcorr_if_needed=True,   # set False if you don't want upsampling
)

print("Zmat shape:", Zmat.shape)
print("Features:", feat_names)

fig_line, fig_heat = plot_zproxy(t, Zmat, feat_names, z_proxy, badframes=badframes)
fig_line.show()
fig_heat.show()

# ----------------------------
# Optional: pick "bad z frames" by robust threshold on z_proxy
# ----------------------------
med = np.nanmedian(z_proxy)
mad = np.nanmedian(np.abs(z_proxy - med)) + 1e-12
thr = med + 6 * mad  # try 4-8
bad_z = z_proxy > thr
bad_z_idx = np.where(bad_z)[0]
print("Likely z/defocus frames (proxy > med+6MAD):", bad_z_idx.size)
print(bad_z_idx)


Zmat shape: (3601, 3)
Features: ['xy_motion_z', 'corrXY_bad_z', 'Vcorr_bad_z_upsampled']


Likely z/defocus frames (proxy > med+6MAD): 18
[  0   1   2   3   5   6   7   8  11  16  20  50  51 106 128 163 175 213]


In [33]:
N_Trace,N_TraceC,N_spikeId,N_VolAX,N_CalAX,N_motor,mask_Voltage,mask_calcium=remove_Frame_Multi(Trace,TraceC,spikeId,VolAX,CalAX,motor)
TracePathCal = os.path.join(path,'calMask.csv')
TracePathVol = os.path.join(path,'volMask.csv')
df = pd.DataFrame(mask_calcium, columns=['mask'])  # create df with column name
df.to_csv(TracePathCal, index=False)
df = pd.DataFrame(mask_Voltage, columns=['mask'])  # create df with column name
df.to_csv(TracePathVol, index=False)

In [34]:
CalAX = N_CalAX
VolTrace = N_Trace
motor = N_motor
TraceC =N_TraceC
VolAX = N_VolAX
spikeId = N_spikeId

In [35]:
spk = sst_spike_correction_gui(VolTrace, fs=500, save_dir=path,name = '',chunk_s=30)

[?] Saved 2612 spikes -> Z:\Adam-Lab-Shared\Data\Michal_Rubin\SRUGC21\X\20-08-20225-ANS\fov2\cell0\final_spikes.pkl


In [36]:
## motor stuff

fig = make_subplots(specs=[[{"secondary_y": True}]])
# svg_path =os.path.join(path,f'EventTypeTrace{n}.svg')
# html_path =os.path.join(path,f'EventTypeTrace{n}.html')

fig.add_trace(go.Scatter(x=VolAX, y=VolTrace, line=dict(color='chocolate', width=0.8),name="Voltage"),secondary_y=False,)

fig.add_trace(go.Scatter(x= CalAX, y=TraceC ,line=dict(color='blue', width=0.8), name="Calcium"),secondary_y=True,)
fig.show()
if motor_active:
    volMot, volRest, calMot, calRest, spikeMot, spikeRest,Change_points,MotIdx,calMotID,RestIdx,calRestId = motorSp(TraceC,VolTrace,motor,CalAX,VolAX,spikeId)
    if np.size(RestIdx[0],0) < 9:
        RestIdx = RestIdx[1:]
        Change_points = Change_points[1:]
        Change_points = Change_points.squeeze()
    changePointPath = os.path.join(path,r'changepoint.csv')
    calMot,calRes,spikeM,spikeR,volM,volR = Split_cal(Change_points,VolTrace,TraceC,VolAX,CalAX,motor,spikeId)
    print(f'calR{len(calRes)}')
    print(f'volR{len(RestIdx)}')
    print(f'calm{len(calMot)}')
    print(f'volm{len(MotIdx)}')
    for i,r in enumerate(calMot):
        #print(f'volMotF{len(MotIdx)}')
        calTracPpathMotor = os.path.join(path, f'calTraceMot{i}.csv')
        pd.DataFrame(r, columns=['trace']).to_csv(calTracPpathMotor, index=False)
        volTracPpathMotor = os.path.join(path, f'volTraceMot{i}.csv')
        pd.DataFrame(volM[i], columns=['trace']).to_csv(volTracPpathMotor, index=False)
        SpiTracPpathMotor = os.path.join(path, f'spikeTraceMot{i}.csv')
        pd.DataFrame(spikeM[i], columns=['trace']).to_csv(SpiTracPpathMotor, index=False)
    for j,m in enumerate(calRes):
        calTracPpathRes = os.path.join(path, f'calTraceRest{j}.csv')
        x = pd.DataFrame(m, columns=['trace']).to_csv(calTracPpathRes, index=False)
        volTracPpathRest = os.path.join(path, f'volTraceRest{j}.csv')
        pd.DataFrame(volR[j], columns=['trace']).to_csv(volTracPpathRest, index=False)
        SpiTracPpathRest = os.path.join(path, f'spikeTraceRest{j}.csv')
        pd.DataFrame(spikeR[j], columns=['trace']).to_csv(SpiTracPpathRest, index=False)
        #print(x.shape)
        print(np.size(m))
    df = pd.DataFrame(np.array(Change_points), columns=['trace'])  # create df with column name
    df.to_csv(changePointPath, index=False)
    # print(len(MotIdx[0]))
    # #print(len(volMot[0]))
    # #print(len(calMot[0]))
    # print(len(RestIdx[0]))
    # print(len(MotIdx[1]))
    # #print(len(volMot[0]))
    # #print(len(calMot[0]))
    # print(len(RestIdx[1]))
    
    #print(len(volRest[0]))
    #print(len(calRest[0]))
    #print(spikeMot)